# TaskAug on PTB-XL — Colab Experiment Runner
**DL4H Spring 2026 | Paul Garcia · Rogelio Medina · Cesar Nava**

Reproduces Raghu et al. (CHIL 2022) TaskAug on PTB-XL (4 tasks: MI, HYP, STTC, CD).

**Runtime:** Google Colab Pro, T4 GPU (16 GB VRAM)  
**Estimated time:** ~20–30 min per task × N combination

---
### Experiment Matrix
| Method | Tasks | N |
|---|---|---|
| No augmentation (baseline) | MI, HYP, STTC, CD | 1000, 5000 |
| TaskAug — frozen policy | MI, HYP, STTC, CD | 1000 |
| TaskAug — global magnitudes | MI, HYP, STTC, CD | 1000 |
| TaskAug — full | MI, HYP, STTC, CD | 1000, 5000 |

Results stored in `results/` as JSON; final table printed at the end.

## 0. Environment Setup

In [1]:
# ── Check GPU ──────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [10]:
# ── Clone fork (idempotent) ────────────────────────────────────────────────
import os, sys
REPO_ROOT = '/content/pyhealth-dl4h-sp26'
if not os.path.isdir(REPO_ROOT):
    !git clone -b feature/pg/taskaug-ecg https://github.com/paulgarciaro/pyhealth-dl4h-sp26.git {REPO_ROOT}
else:
    print('Repo already cloned, pulling latest ...')
    !git -C {REPO_ROOT} pull --ff-only

# ── Add repo to path (editable-style, no pip install needed) ─────────────
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# ── Install only the lightweight runtime deps ─────────────────────────────
!pip install wfdb scipy scikit-learn pandas tqdm polars pyarrow pydantic \
             platformdirs mne litdata filelock dask distributed narwhals \
             more-itertools einops -q

# ── Verify our files are present ─────────────────────────────────────────
assert os.path.isfile(f'{REPO_ROOT}/pyhealth/datasets/ptbxl.py'), 'ptbxl.py missing!'
assert os.path.isfile(f'{REPO_ROOT}/pyhealth/models/taskaug_resnet.py'), 'taskaug_resnet.py missing!'
print('Fork verified ✓  REPO_ROOT =', REPO_ROOT)

Cloning into 'pyhealth-dl4h-sp26'...
remote: Enumerating objects: 9976, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 9976 (delta 10), reused 22 (delta 8), pack-reused 9945 (from 1)
Receiving objects: 100% (9976/9976), 130.94 MiB | 23.44 MiB/s, done.
Resolving deltas: 100% (6340/6340), done.
/content/pyhealth-dl4h-sp26/pyhealth-dl4h-sp26/pyhealth-dl4h-sp26
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pyhealth (pyproject.toml) ... done
pyhealth loaded from: /content/pyhealth-dl4h-sp26/pyhealth-dl4h-sp26/pyhealth-dl4h-sp26/pyhealth/__init__.py
PTBXLDataset.prepare_metadata found ✓


In [4]:
# ── Mount Google Drive (store PTB-XL + results there) ─────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/DL4H_SP26'
PTB_XL_ROOT = f'{DRIVE_ROOT}/ptb-xl-1.0.3'   # ← set to your PTB-XL path
RESULTS_DIR = f'{DRIVE_ROOT}/results'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print('PTB-XL root:', PTB_XL_ROOT)
print('Results dir:', RESULTS_DIR)

Mounted at /content/drive
PTB-XL root: /content/drive/MyDrive/DL4H_SP26/ptb-xl-1.0.3
Results dir: /content/drive/MyDrive/DL4H_SP26/results


In [5]:
# ── Download PTB-XL (skip if already on Drive) ─────────────────────────────
import os

if not os.path.isfile(f'{PTB_XL_ROOT}/ptbxl_database.csv'):
    print('Downloading PTB-XL metadata ...')
    os.makedirs(PTB_XL_ROOT, exist_ok=True)
    !wget -q -P {PTB_XL_ROOT} https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv
    !wget -q -P {PTB_XL_ROOT} https://physionet.org/files/ptb-xl/1.0.3/scp_statements.csv
    print('Downloading waveforms (records500, ~1.8 GB) ...')
    !wget -q -r -nH --cut-dirs=3 -P {PTB_XL_ROOT} \
        https://physionet.org/files/ptb-xl/1.0.3/records500/
    print('Download complete.')
else:
    print('PTB-XL already present, skipping download.')

^C
Download complete.


## 1. Dataset & Shared Utilities

In [11]:
import json, time, warnings
import numpy as np
import torch
from pathlib import Path

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

# ── Build PTB-XL metadata once ─────────────────────────────────────────────
from pyhealth.datasets.ptbxl import PTBXLDataset

if not os.path.isfile(f'{PTB_XL_ROOT}/ptbxl-pyhealth.csv'):
    print('Building ptbxl-pyhealth.csv ...')
    PTBXLDataset.prepare_metadata(PTB_XL_ROOT)

print('Metadata ready.')

Using device: cuda
Building ptbxl-pyhealth.csv ...
Wrote 21,799 records to /content/drive/MyDrive/DL4H_SP26/ptb-xl-1.0.3/ptbxl-pyhealth.csv (MI=5469, HYP=2649, STTC=5235, CD=4898)
Wrote 21,799 records to /content/drive/MyDrive/DL4H_SP26/ptb-xl-1.0.3/ptbxl-pyhealth.csv (MI=5469, HYP=2649, STTC=5235, CD=4898)


INFO:pyhealth.datasets.ptbxl:Wrote 21,799 records to /content/drive/MyDrive/DL4H_SP26/ptb-xl-1.0.3/ptbxl-pyhealth.csv (MI=5469, HYP=2649, STTC=5235, CD=4898)


Metadata ready.


In [12]:
import importlib.util as _ilu

# ── pyhealth sub-packages that are safe to import directly ───────────────
from pyhealth.datasets import get_dataloader, split_by_patient
from pyhealth.datasets.ptbxl import PTBXLDataset
from pyhealth.tasks.ecg_classification_ptbxl import ECGBinaryClassificationPTBXL
from pyhealth.metrics import binary_metrics_fn

# ── Load taskaug_resnet directly to skip pyhealth/models/__init__.py ─────
# models/__init__.py imports MoleRec (rdkit), TransformerDeID (transformers),
# GAMENet (torch_geometric), etc. — none of which are needed here.
_spec = _ilu.spec_from_file_location(
    'taskaug_resnet',
    f'{REPO_ROOT}/pyhealth/models/taskaug_resnet.py'
)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
ResNet1D      = _mod.ResNet1D
TaskAugPolicy = _mod.TaskAugPolicy
BiLevelTrainer = _mod.BiLevelTrainer
print('ResNet1D / TaskAugPolicy / BiLevelTrainer loaded ✓')

TASKS   = ['MI', 'HYP', 'STTC', 'CD']
N_SIZES = [1000, 5000]
N_SPLITS = 15        # paper uses 15 random 80/10/10 splits
EPOCHS   = 50
BATCH    = 64


def build_loaders(task: str, n: int, seed: int, ptb_root: str):
    """Return (train_loader, val_loader, test_loader) for one split."""
    dataset = PTBXLDataset(root=ptb_root, dev=False)
    task_fn = ECGBinaryClassificationPTBXL(task=task)
    sample_ds = dataset.set_task(task_fn)

    # patient-level split
    train_ds, val_ds, test_ds = split_by_patient(
        sample_ds, [0.8, 0.1, 0.1], seed=seed
    )

    # sub-sample training fold to n patients
    rng = np.random.default_rng(seed)
    pids = list(set(s['patient_id'] for s in train_ds))
    chosen = set(rng.choice(pids, size=min(n, len(pids)), replace=False))
    train_ds = [s for s in train_ds if s['patient_id'] in chosen]

    train_loader = get_dataloader(train_ds, batch_size=BATCH, shuffle=True)
    val_loader   = get_dataloader(val_ds,   batch_size=BATCH, shuffle=False)
    test_loader  = get_dataloader(test_ds,  batch_size=BATCH, shuffle=False)
    return train_loader, val_loader, test_loader, sample_ds


def run_split(task, n, seed, method, ptb_root, output_base):
    """Train one method on one split; return {roc_auc, pr_auc}."""
    train_loader, val_loader, test_loader, sample_ds = build_loaders(
        task, n, seed, ptb_root
    )
    model  = ResNet1D(dataset=sample_ds, feature_keys=['signal'],
                      label_key='label', mode='binary').to(DEVICE)
    policy = TaskAugPolicy(n_stages=2, n_ops=7).to(DEVICE)

    # Method-specific policy configuration
    if method == 'no_aug':
        # Train model only; no augmentation applied
        from torch.optim import Adam
        opt = Adam(model.parameters(), lr=1e-3)
        model.train()
        for epoch in range(EPOCHS):
            for batch in train_loader:
                batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v
                         for k, v in batch.items()}
                loss = model(**batch)['loss']
                opt.zero_grad(); loss.backward(); opt.step()
        trainer = BiLevelTrainer(model=model, policy=policy, device=DEVICE,
                                 output_path=output_base,
                                 exp_name=f'{method}_{task}_n{n}_s{seed}')

    elif method == 'frozen_policy':
        # Policy initialized but NOT updated (frozen)
        for p in policy.parameters():
            p.requires_grad_(False)
        trainer = BiLevelTrainer(model=model, policy=policy, device=DEVICE,
                                 output_path=output_base,
                                 exp_name=f'{method}_{task}_n{n}_s{seed}')
        trainer.train(train_loader, val_loader, epochs=EPOCHS,
                      monitor='roc_auc')

    elif method == 'global_mag':
        # Class-specific magnitudes disabled: tie neg=pos by sharing one param
        with torch.no_grad():
            policy.magnitudes_pos.copy_(policy.magnitudes_neg)
        # Prevent pos from diverging by zeroing its grad during outer loop
        trainer = BiLevelTrainer(model=model, policy=policy, device=DEVICE,
                                 output_path=output_base,
                                 exp_name=f'{method}_{task}_n{n}_s{seed}')
        trainer.train(train_loader, val_loader, epochs=EPOCHS,
                      monitor='roc_auc')

    else:  # 'taskaug' — full method
        trainer = BiLevelTrainer(model=model, policy=policy, device=DEVICE,
                                 output_path=output_base,
                                 exp_name=f'{method}_{task}_n{n}_s{seed}')
        trainer.train(train_loader, val_loader, epochs=EPOCHS,
                      inner_lr=1e-3, outer_lr=1e-2, neumann_order=3,
                      monitor='roc_auc')

    y_true, y_prob, _ = trainer.inference(test_loader)
    return binary_metrics_fn(y_true, y_prob, metrics=['roc_auc', 'pr_auc'])


print('Utilities loaded.')

ModuleNotFoundError: No module named 'rdkit'

## 2. Experiment Runner
Each cell below runs one method × task × N combination across 15 splits and saves results to Drive.

In [ ]:
def run_experiment(method: str, tasks=TASKS, n_sizes=N_SIZES,
                   n_splits=N_SPLITS, ptb_root=PTB_XL_ROOT, results_dir=RESULTS_DIR):
    """Run method across all tasks, N sizes, and splits. Save JSON results."""
    all_results = {}

    for task in tasks:
        for n in n_sizes:
            key = f'{method}__{task}__n{n}'
            result_file = f'{results_dir}/{key}.json'

            if os.path.isfile(result_file):
                print(f'[SKIP] {key} — already done')
                with open(result_file) as f:
                    all_results[key] = json.load(f)
                continue

            split_scores = []
            print(f'\n>>> {key}')
            for seed in range(n_splits):
                t0 = time.time()
                scores = run_split(task, n, seed, method, ptb_root,
                                   f'{results_dir}/ckpts')
                elapsed = time.time() - t0
                split_scores.append(scores)
                print(f'  split {seed:02d} | AUROC={scores["roc_auc"]:.4f} '
                      f'AUPRC={scores["pr_auc"]:.4f} | {elapsed:.0f}s')

            agg = {
                'roc_auc_mean': float(np.mean([s['roc_auc'] for s in split_scores])),
                'roc_auc_std':  float(np.std( [s['roc_auc'] for s in split_scores])),
                'pr_auc_mean':  float(np.mean([s['pr_auc']  for s in split_scores])),
                'pr_auc_std':   float(np.std( [s['pr_auc']  for s in split_scores])),
                'splits': split_scores,
            }
            all_results[key] = agg
            with open(result_file, 'w') as f:
                json.dump(agg, f, indent=2)
            print(f'  → AUROC {agg["roc_auc_mean"]:.4f} ± {agg["roc_auc_std"]:.4f} '
                  f'| AUPRC {agg["pr_auc_mean"]:.4f} ± {agg["pr_auc_std"]:.4f}')

    return all_results

print('run_experiment() ready.')

### 2a. No Augmentation Baseline

In [ ]:
results_no_aug = run_experiment('no_aug')

### 2b. TaskAug — Frozen Policy (Ablation)

In [ ]:
# Only need N=1000 for ablations (matches paper Table 3)
results_frozen = run_experiment('frozen_policy', n_sizes=[1000])

### 2c. TaskAug — Global Magnitudes (Ablation)

In [ ]:
results_global = run_experiment('global_mag', n_sizes=[1000])

### 2d. TaskAug — Full Method

In [ ]:
results_taskaug = run_experiment('taskaug')

## 3. Results Table

In [ ]:
import pandas as pd

def load_result(method, task, n, results_dir=RESULTS_DIR):
    path = f'{results_dir}/{method}__{task}__n{n}.json'
    if not os.path.isfile(path):
        return None
    with open(path) as f:
        return json.load(f)


def fmt(r, metric='roc_auc'):
    if r is None:
        return 'TBD'
    return f"{r[f'{metric}_mean']:.3f} ± {r[f'{metric}_std']:.3f}"


methods = [
    ('no_aug',       'No augmentation'),
    ('frozen_policy','TaskAug frozen policy'),
    ('global_mag',   'TaskAug global magnitudes'),
    ('taskaug',      'TaskAug full'),
]

for n in N_SIZES:
    print(f'\n=== AUROC (N={n}) ===')
    rows = []
    for method_key, method_label in methods:
        if n == 5000 and method_key in ('frozen_policy', 'global_mag'):
            continue
        row = {'Method': method_label}
        for task in TASKS:
            r = load_result(method_key, task, n)
            row[task] = fmt(r, 'roc_auc')
        rows.append(row)
    print(pd.DataFrame(rows).to_string(index=False))

    print(f'\n=== AUPRC (N={n}) ===')
    rows = []
    for method_key, method_label in methods:
        if n == 5000 and method_key in ('frozen_policy', 'global_mag'):
            continue
        row = {'Method': method_label}
        for task in TASKS:
            r = load_result(method_key, task, n)
            row[task] = fmt(r, 'pr_auc')
        rows.append(row)
    print(pd.DataFrame(rows).to_string(index=False))

## 4. Save Final Results Summary

In [ ]:
# Consolidate all JSON result files into one summary
summary = {}
for method_key, _ in methods:
    for task in TASKS:
        for n in N_SIZES:
            r = load_result(method_key, task, n)
            if r:
                summary[f'{method_key}__{task}__n{n}'] = {
                    'roc_auc': f"{r['roc_auc_mean']:.4f}",
                    'pr_auc':  f"{r['pr_auc_mean']:.4f}",
                }

summary_path = f'{RESULTS_DIR}/all_results_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Summary saved to {summary_path}')
print(json.dumps(summary, indent=2))